# Stage 2 Notebook 12 - Exp2G RMT-GCA + CLRKD-style lane head

Exp2E and Exp2F plateaued at val/lane_point_mae ~0.40 because every one of the 192 lane priors received the same global-pool feature and could differentiate only through a 3-d prior embedding. Per-prior gradients were nearly identical across the batch.

Exp2G replaces the lane head with a CLRKD-style ROI gather:

- per-prior, per-scale bilinear sampling along each prior's own current curve (gradients flow through sample positions),
- per-scale 1-D conv over the sample-points axis,
- multi-scale fusion,
- cross-attention to the full feature map,
- 3 refinement stages with auxiliary supervision on every intermediate stage.

Backbone (RMT + GCA), detection head (DETR), dynamic-k matching, focal classification, geometry warmup, and val metrics are unchanged from Exp2E so the comparison isolates the lane-head architecture.

Reference: external_repos/CLRNet/clrnet/models/utils/roi_gather.py and external_repos/CLRKDNet-master/clrkd/models/heads/clr_head.py.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run so logs appear quickly.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file under `stage2/logs/`.
4. The dataset tar is read from Drive; the curve dataset preparation in Notebook 00 should not be rerun unless the tar is missing or corrupted.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp07_rmt_gca_clrkd_lane_head_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: tiny forward + backward over the new CLRKDLaneHead. This must
# print 'OK exp07_*.yaml' with shapes before training is attempted. If the
# smoke test fails, do NOT proceed to training.
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp07_rmt_gca_clrkd_lane_head_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp07_rmt_gca_clrkd_lane_head_joint_smoke.log
OK exp07_rmt_gca_clrkd_lane_head_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.7031 det_loss=3.3345 grad_cos=-0.0659 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5001022815704346, 'gate/lane_mean': 0.5010365843772888, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp07_rmt_gca_clrkd_lane_head_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

# First run should be a small debug run. Change to False only after the
# smoke cell above succeeded and per-epoch metrics look normal.
DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp07_rmt_gca_clrkd_lane_head_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp07_rmt_gca_clrkd_lane_head_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp07_rmt_gca_clrkd_lane_head_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp07_rmt_gca_clrkd_lane_head_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp07_rmt_gca_clrkd_lane_head_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp07_rmt_gca_clrkd_lane_head_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd

0

## What to watch during Exp2G training

Compare against Exp2E (final epoch 10): val/lane_point_mae 0.401, val/matched_line_iou ~0.08, val/lane_exist_best_f1 ~0.71 with precision ~ recall ~ 0.55 (no real separation). Exp2G should improve the geometry and existence calibration if the architectural fix is correct.

Strong signals that Exp2G works:

- `val/lane_point_mae`: monotonically decreases from epoch 1 onward and ends below 0.34. A flat plateau at 0.40 means the new head is still bottlenecked.
- `val/matched_line_iou`: rises from <0.10 to >=0.20 by epoch 10. This is the meaningful geometry signal; the loss-shape `val/lane/line_iou` is misleading because background priors dominate the average.
- `val/lane_exist_best_f1` >= 0.75 with both `val/lane_exist_precision` and `val/lane_exist_recall` >= 0.65. Existence calibration is real, not threshold-tuning luck.
- `val/lane_exist_pos_score_mean` minus `val/lane_exist_neg_score_mean` >= 0.20. Positive and negative priors are actually separated in score space.
- Auxiliary losses `train/lane/aux0_total`, `train/lane/aux1_total` (one per intermediate refinement stage) should drop together with `train/lane/total` instead of plateauing.
- `val/det/metric_map50` does not regress more than 0.02 from Exp2B.

Failure signals that mean another approach is needed:

- Geometry plateaus and matched_line_iou < 0.12 after 10 epochs - prior init scheme or geometry parameterization is the deeper issue, not the per-prior feature pathway.
- Existence F1 stays below 0.65 despite point_mae improving - try asymmetric focal or hard-negative mining over unmatched priors.
- Detection mAP50 regresses sharply - GCA gates may saturate under the heavier lane branch; consider gate regularization (`loss.lambda_gate_reg`) or reducing lane branch LR.

After short10 finishes, run Notebook 08 to plot per-epoch trends and produce the box+lane preview video.